In [1]:
import sys
import os
if os.path.abspath('..') not in sys.path:
    sys.path.append(os.path.abspath('..'))
import os
import pandas as pd
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall
)

# Kendi RAG sistemimizi içeri aktarıyoruz
from src.agents.graph import create_agent_graph

app = create_agent_graph()

# ---------------------------------------------------------
# 1. 20 SORULUK TEST VERİ SETİ (GROUND TRUTH) HAZIRLIĞI
# ---------------------------------------------------------
# Sistemin farklı metadataları, farklı şirketleri ve 
# karşılaştırma yeteneklerini test eden kapsamlı liste
questions = [
    # Temel Par Value (Nominal Değer) Soruları
    "What is the par value of common stock for Tesla?",
    "What is the par value of common stock for Nvidia?",
    "What is the par value of common stock for Amazon?",
    "What is the par value of common stock for Netflix?",
    "What is the par value of common stock for Intel?",
    "What is the par value of common stock for Bank of America?",
    "What is the par value of common stock for Johnson & Johnson?",
    "What is the par value of Class A common stock for Meta Platforms?",
    
    # Şirket Karşılaştırmaları (Multi-Hop / Multi-Document Retrieval)
    "Compare the par value of common stock for Apple and Tesla.",
    "Compare the par value of common stock for Intel and AMD.",
    "Which company has a smaller par value, Meta or Amazon?",
    "Compare the state of incorporation for Apple and Microsoft.",
    
    # Kuruluş Eyaleti (State of Incorporation) Soruları
    "In which state is Microsoft incorporated?",
    "In which state is Meta Platforms incorporated?",
    "In which state is Walmart incorporated?",
    "In which state is Nvidia incorporated?",
    
    # Resmi Şirket Adı (Exact Name of Registrant) Soruları
    "What is the exact name of the registrant for ticker GOOGL?",
    "What is the exact name of the registrant for ticker JPM?",
    "What is the exact name of the registrant for ticker DIS?",
    
    # Tercihli Hisse (Preferred Stock) Detayları
    "Does Tesla have any preferred stock, and if so, what is its par value?"
]

# RAGAS Hakeminin "Doğru Cevap" olarak kabul edeceği referans cevaplar
ground_truths = [
    "Tesla's common stock has a par value of $0.001 per share.",
    "Nvidia's common stock has a par value of $0.001 per share.",
    "Amazon's common stock has a par value of $0.01 per share.",
    "Netflix's common stock has a par value of $0.001 per share.",
    "Intel's common stock has a par value of $0.001 per share.",
    "Bank of America's common stock has a par value of $0.01 per share.",
    "Johnson & Johnson's common stock has a par value of $1.00 per share.",
    "Meta's Class A common stock has a par value of $0.000006 per share.",
    
    "Apple's par value is $0.00001, while Tesla's par value is $0.001.",
    "Intel's par value is $0.001, while AMD's par value is $0.01.",
    "Meta has a smaller par value ($0.000006) compared to Amazon ($0.01).",
    "Apple is incorporated in California, while Microsoft is incorporated in Washington.",
    
    "Microsoft is incorporated in the State of Washington.",
    "Meta Platforms is incorporated in Delaware.",
    "Walmart is incorporated in the State of Delaware.",
    "Nvidia is incorporated in the State of Delaware.",
    
    "The exact name of the registrant is Alphabet Inc.",
    "The exact name of the registrant is JPMorgan Chase & Co.",
    "The exact name of the registrant is The Walt Disney Company.",
    
    "Yes, Tesla has preferred stock with a par value of $0.001 per share."
]

# ---------------------------------------------------------
# 2. SİSTEMİ ÇALIŞTIRIP CEVAPLARI TOPLAMA
# ---------------------------------------------------------
answers = []
contexts = []

print("🚀 RAG Sistemi 20 soruluk stres testine başlıyor...")
print("⚠️ Lütfen bekleyin, bu işlem (20 soru x Ajan Döngüleri) birkaç dakika sürebilir...\n")

for i, q in enumerate(questions):
    print(f"[{i+1}/20] Soru: {q}")
    
    # Ajan sistemini tetikle
    result = app.invoke({"question": q})
    answers.append(result["final_answer"])
    
    # Qdrant'tan dönen belgeleri RAGAS formatına alıyoruz
    docs = result.get("documents", [])
    contexts.append(docs)
    
    print(f"     ✅ Cevaplandı.\n")

# ---------------------------------------------------------
# 3. RAGAS DATASET FORMATINA ÇEVİRME
# ---------------------------------------------------------
data = {
    "question": questions,
    "answer": answers,
    "contexts": contexts,
    "ground_truth": ground_truths
}
dataset = Dataset.from_dict(data)

# ---------------------------------------------------------
# 4. KARNE ÇIKARMA (EVALUATION)
# ---------------------------------------------------------
# YENİ EKLENEN KÜTÜPHANELER (Hakem modelleri için)
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# ---------------------------------------------------------
# 4. KARNE ÇIKARMA (EVALUATION) - GÜNCELLENDİ
# ---------------------------------------------------------
print("📊 Tüm cevaplar toplandı. RAGAS Hakem Ajanları açık modellerle devreye giriyor...")
print("⏳ Karne hazırlanıyor, bu işlem OpenAI API'sini kullanır ve biraz vakit alabilir...\n")

# Hata almamak için RAGAS'a "kendi kafana göre değil, bu modelleri kullan" diyoruz
evaluator_llm = ChatOpenAI(model="gpt-4o-mini")
evaluator_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

evaluation_result = evaluate(
    dataset=dataset,
    metrics=[
        context_precision,
        context_recall,
        faithfulness,
        answer_relevancy,
    ],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings
)

# Sonuçları DataFrame'e çevirip ekrana basalım
df_result = evaluation_result.to_pandas()
print("\n🎉 İŞTE RAG SİSTEMİNİN NİHAİ KARNESİ:\n")
display(df_result)

# Sonuçları CSV olarak kaydetmek istersen:
df_result.to_csv("ragas_evaluation_results.csv", index=False)
print("💾 Sonuçlar 'ragas_evaluation_results.csv' dosyasına kaydedildi.")

/var/folders/mk/jj0k081x35zcyxnhns8rfzq00000gn/T/ipykernel_96182/3950064165.py:9: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
/var/folders/mk/jj0k081x35zcyxnhns8rfzq00000gn/T/ipykernel_96182/3950064165.py:9: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
/var/folders/mk/jj0k081x35zcyxnhns8rfzq00000gn/T/ipykernel_96182/3950064165.py:9: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
/

⏳ Local Embedding modeli yükleniyor (İlk çalışmada modeli indirir)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

🚀 RAG Sistemi 20 soruluk stres testine başlıyor...
⚠️ Lütfen bekleyin, bu işlem (20 soru x Ajan Döngüleri) birkaç dakika sürebilir...

[1/20] Soru: What is the par value of common stock for Tesla?
🧠 Planner: Soru analiz ediliyor, plan ve hedef şirket çıkarılıyor...
🎯 Hedef Şirket Tespit Edildi: TSLA
📚 Retriever: Şu şirketler aranıyor: ['TSLA']
🔍 Uygulanan filtre ile arama yapılıyor...
🌐 Web Researcher: Qdrant yetersiz kaldı, internet araştırması başlatılıyor...
🌐 Web Researcher: İnternet araştırması tamamlandı ve 'Resmi Belgeler' arasına eklendi.
✍️ Synthesizer: Veriler birleştiriliyor (Deneme: 1)...
🕵️‍♂️ Validator: Rakamlar GERÇEK kaynaklarda aranıyor...
✅ Validator: Tüm sayılar doğru, onaylandı!
     ✅ Cevaplandı.

[2/20] Soru: What is the par value of common stock for Nvidia?
🧠 Planner: Soru analiz ediliyor, plan ve hedef şirket çıkarılıyor...
🎯 Hedef Şirket Tespit Edildi: NVDA
📚 Retriever: Şu şirketler aranıyor: ['NVDA']
🔍 Uygulanan filtre ile arama yapılıyor...
🌐 Web Researcher: 

Evaluating:   0%|          | 0/80 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.



🎉 İŞTE RAG SİSTEMİNİN NİHAİ KARNESİ:



,user_input,retrieved_contexts,response,reference,context_precision,context_recall,faithfulness,answer_relevancy
0,What is the par value of common stock for Tesla?,"[""us-gaap_CommonStockSharesOutstanding"": {\n ...",The par value of common stock for Tesla is $0....,Tesla's common stock has a par value of $0.001...,0.757576,1.0,1.0,1.000000
1,What is the par value of common stock for Nvidia?,"[colspan=""3"" style=""background-color:#ffffff;p...",The par value of common stock for Nvidia is $0...,Nvidia's common stock has a par value of $0.00...,0.340909,1.0,1.0,1.000000
2,What is the par value of common stock for Amazon?,"[""calculation"": {\n ""http://www.amazon.co...",The par value of common stock for Amazon is $0...,Amazon's common stock has a par value of $0.01...,0.252814,1.0,1.0,1.000000
3,What is the par value of common stock for Netf...,"[sale of up to <ix:nonFraction unitRef=""shares...",The par value of common stock for Netflix is $...,Netflix's common stock has a par value of $0.0...,0.331650,1.0,1.0,1.000000
4,What is the par value of common stock for Intel?,"[""calculation"": {\n ""http://www.intel.com...",The par value of common stock for Intel Corpor...,Intel's common stock has a par value of $0.001...,0.590909,1.0,1.0,0.980279
5,What is the par value of common stock for Bank...,"[""weight"": 1.0,\n ""order"": 1.0\n }\...",The par value of common stock for Bank of Amer...,Bank of America's common stock has a par value...,0.205195,1.0,1.0,1.000000
6,What is the par value of common stock for John...,[par value $0.01 per share (the Kenvue Common ...,The par value of common stock for Johnson & Jo...,Johnson & Johnson's common stock has a par val...,0.153409,1.0,1.0,1.000000
7,What is the par value of Class A common stock ...,[Class&#160;A common stock and <ix:nonFraction...,The par value of Class A common stock for Meta...,Meta's Class A common stock has a par value of...,0.493403,1.0,1.0,0.980051
8,Compare the par value of common stock for Appl...,"[format=""ixt:num-dot-decimal"" scale=""0"" id=""f-...",Apple's common stock has a par value of $0.000...,"Apple's par value is $0.00001, while Tesla's p...",0.090909,1.0,0.5,0.874459
9,Compare the par value of common stock for Inte...,"[""r1152""\n ]\n },\n ""us-gaap_CommonS...",The par value of common stock for Intel is $0....,"Intel's par value is $0.001, while AMD's par v...",0.090909,1.0,1.0,0.867490


💾 Sonuçlar 'ragas_evaluation_results.csv' dosyasına kaydedildi.
